[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mauriciorslrv/DS_basics/blob/main/00%20Fundamentos%20de%20Datos%20-%20Excel%20y%20SQL/notebooks/02_SQL_desde_cero.ipynb)

# 00.02 · SQL desde cero: consulta, une y decide

Este notebook usa la biblioteca estándar **SQLite** para permitir primeras consultas sin servidor. **PostgreSQL es el motor de referencia para el proyecto**: ejecuta los SQL de la carpeta `sql/` para el modelo PostgreSQL. Algunas funciones sintácticas difieren entre motores.

In [ ]:
# Lectura portátil: localmente se usan los CSV del repo; en Colab se descargan los mismos datos públicos sintéticos.
from pathlib import Path
_data = Path('00 Fundamentos de Datos - Excel y SQL/datos')
if not _data.exists() and not Path('../datos/ventas.csv').exists() and not Path('../../00 Fundamentos de Datos - Excel y SQL/datos/ventas.csv').exists():
    from urllib.request import urlretrieve
    _data.mkdir(parents=True, exist_ok=True)
    for _file in ['ventas.csv','clientes.csv','productos.csv']:
        urlretrieve('https://raw.githubusercontent.com/mauriciorslrv/DS_basics/main/00%20Fundamentos%20de%20Datos%20-%20Excel%20y%20SQL/datos/' + _file, _data / _file)
print('Datos listos para este ejercicio.')


In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
base=Path('../datos') if Path('../datos/ventas.csv').exists() else Path('00 Fundamentos de Datos - Excel y SQL/datos')
con=sqlite3.connect(':memory:')
for tabla in ['clientes','productos','ventas']:
    data=pd.read_csv(base/f'{tabla}.csv')
    data.to_sql(tabla,con,index=False,if_exists='replace')
print(pd.read_sql_query('SELECT COUNT(*) AS ventas FROM ventas',con))

## Lección 1 · SELECT, WHERE, ORDER BY, LIMIT
Tarea: detectar compradores de Canadá, comparar cantidades y ordenar por fecha. Reflexiona sobre el sesgo de usar solo operaciones recientes.

In [ ]:
q='''SELECT v.id,v.fecha,c.nombre,c.pais,v.unidades FROM ventas v JOIN clientes c ON v.cliente_id=c.cliente_id WHERE c.pais='CA' ORDER BY v.fecha DESC LIMIT 10'''
pd.read_sql_query(q,con)

## Lección 2 · GROUP BY y HAVING
¿Por qué debemos sumar ingresos y costos, en lugar de promediar porcentajes por fila?

In [ ]:
q='''SELECT c.pais,COUNT(*) AS pedidos,SUM(v.unidades*v.precio_unitario) AS ingresos,SUM(v.unidades*(v.precio_unitario-v.costo_unitario)) AS margen FROM ventas v JOIN clientes c ON v.cliente_id=c.cliente_id GROUP BY c.pais HAVING COUNT(*) >= 5 ORDER BY margen DESC'''
pd.read_sql_query(q,con)

## Lección 3 · Relaciones y JOIN
Construye un diagrama simple: `clientes (1) → ventas (N) ← (1) productos`. Prueba INNER y LEFT JOIN. ¿Qué pasa si un cliente no ha comprado?

In [ ]:
q='''SELECT c.cliente_id,c.nombre,COUNT(v.id) AS numero_compras FROM clientes c LEFT JOIN ventas v ON v.cliente_id=c.cliente_id GROUP BY c.cliente_id,c.nombre ORDER BY numero_compras'''
pd.read_sql_query(q,con)

## Lección 4 · CTE y funciones de ventana
Ranking de compradores por ingresos y participación acumulada. El objetivo no es aplicar reglas 80/20 sin evidencia: verifica el porcentaje real.

In [ ]:
q='''WITH totales AS (SELECT c.nombre, SUM(v.unidades*v.precio_unitario) AS ingreso FROM clientes c JOIN ventas v ON c.cliente_id=v.cliente_id GROUP BY c.nombre) SELECT nombre,ingreso,ROUND(100.0*ingreso/SUM(ingreso) OVER(),2) AS participacion_pct, ROUND(100.0*SUM(ingreso) OVER(ORDER BY ingreso DESC)/SUM(ingreso) OVER(),2) AS acumulado_pct FROM totales ORDER BY ingreso DESC'''
pd.read_sql_query(q,con)

## Reto integrador (extenso)
1. Diseña una consulta que dé margen mensual por producto y mercado.
2. Calcula los tres clientes con más ventas y frecuencia de compra.
3. Simula incremento de costo 8% en un producto y decide si revisar precio.
4. Repite en PostgreSQL e identifica diferencias de dialecto.
5. Explica supuestos y brechas de datos: logística, aranceles, gastos fijos, incidencias.
6. Produce un documento para dirección con conclusiones sustentadas.

**Opcional:** implementa restricciones, PK/FK e índices en PostgreSQL y comprueba el plan con EXPLAIN.